# DINGO Tutorial

**Welcome to the DINGO tutorial!**

*Tutorial by Annalena Kofler ([original repository](https://github.com/annalena-k/tutorial-dingo-introduction)); updated for `dingo-gw` 0.10.3 and DINGO-T1 in November 2026.*

This notebook provides an introduction to the DINGO machine-learning package for gravitational wave (GW) parameter estimation. It shows how to train a DINGO model from scratch, or use an already-trained model to obtain posterior samples for GW data. We hope that this will serve as a starting point for those interested in using DINGO.

For a more general introduction to neural posterior estimation of gravitational waves (without DINGO), see the tutorial ["GW Parameter Inference with Machine Learning"](https://github.com/stephengreen/gw-school-corfu-2023).

Since it takes several days to train a production-level DINGO model (14 GW parameters, ~100 million network parameters, ~10 million training examples), we make several simplifications:
- Restrict the parameter space to two parameters (chirp mass and the mass ratio).
- Use a much smaller network and training dataset.
This will give reduced performance, but can be run in real time.

## Tutorial Structure
1. Generating a waveform dataset
2. Generating a noise dataset
3. Training
4. Inference on injections
5. Inference on a real event with `dingo_pipe`
6. Inference with the pre-trained DINGO-T1 model (from Zenodo)

Update 13.11.2026: Updated for `dingo-gw==0.10.3`. This version has been tested locally; it has not yet been tested on Google Colab.

The settings below run on a CPU (`device: cpu`). To use a GPU (on Colab: `Runtime` -> `Change runtime type` -> GPU), change `device` to `cuda` in the training and inference settings.

## 0. Installation

We use `pip`, but dingo can also be installed using `conda`. The file `constraints.txt` pins the dependencies to the versions `dingo-gw==0.10.3` was tested with; in particular, newer versions of `bilby_pipe` are not yet compatible with `dingo_pipe`. On Colab, the cell below installs everything. For a local installation, see the README.

In [ ]:
%%capture
import sys
if "google.colab" in sys.modules:
    # Pin dependencies to the versions dingo-gw 0.10.3 was tested with (newer bilby_pipe breaks dingo_pipe).
    !pip3 install dingo-gw==0.10.3 -c https://raw.githubusercontent.com/stephengreen/dingo-tutorial/main/constraints.txt

To check whether `dingo` is correctly installed and ready to use, run:

In [ ]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

import dingo
print(dingo.__version__) # Should be >= 0.10.3

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import yaml

Prepare folder structure for tutorial:

In [ ]:
# Remove automatically created folder 'sample_data'
if os.path.isdir("sample_data"):
    os.system("rm -rf sample_data")
# Create folders required for the tutorial
os.makedirs('01_training_data/asd_dataset', exist_ok=True)
os.makedirs('01_training_data/waveform_dataset', exist_ok=True)
os.makedirs('02_training', exist_ok=True)
os.makedirs('03_inference/injection', exist_ok=True)
os.makedirs('04_exercise/lum_dist_marginalization', exist_ok=True)
os.makedirs('04_exercise/with_lum_dist', exist_ok=True)
os.makedirs('05_pretrained_model', exist_ok=True)

## 1. Generating a waveform dataset
**Step 1: Prepare settings file**

DINGO uses YAML files to configure all settings for generating training data and training models.

A waveform dataset settings file specifies the following:
- **domain**: Supported domain types are `UniformFrequencyDomain` and `MultibandedFrequencyDomain`. Specifies, e.g., frequency domain within the range $[f_\mathrm{min}, f_\mathrm{max}]$ and $\delta f$. DINGO currently does not support training on time domain directly. Data from time-domain waveform generators is transformed and saved in the frequency domain.
- **waveform_generator**: Specify waveform approximant (e.g., IMRPhenomPv2, IMRPhenomXPHM, SEOBNRv4PHM, SEOBNRv5PHM) and reference frequency. Note that SEOBNRv5 waveforms require `pyseobnr` to be installed separately (optional).
- **intrinsic_prior**: The parameter space is split into intrinsic and extrinsic components: *Intrinsic* parameters refer to those that are needed to generate waveform polarizations. *Extrinsic* parameters refer to those parameters that can be sampled and applied rapidly during training. Luminosity distance and time of coalescence are considered as both intrinsic and extrinsic. They are needed to generate polarizations, but they can also be easily transformed during training to augment the dataset. We therefore fix them to fiducial values for generating polarizations.
- **num_samples**: Number of waveforms to generate. In a production setting, it is recommended to train a model with (at least) $5 \cdot 10^6$ waveforms, with larger parameter space requiring more samples.
- **compression**: For large datasets, it is recommended to save compressed waveforms using a [singular-value decomposition (SVD)](https://en.wikipedia.org/wiki/Singular_value_decomposition). Since we only generate a dataset with $10^4$ waveforms, we do not apply the compression. Details about the SVD settings can be found [here](https://dingo-gw.readthedocs.io/en/latest/waveform_dataset.html#generating-a-simple-dataset).

*For this tutorial, we analyze only chirp mass and the mass ratio, fixing all other parameters*

More information about generating the waveform dataset can be found in the sections [Generating Waveforms](https://dingo-gw.readthedocs.io/en/latest/generating_waveforms.html) and [Building a waveform dataset](https://dingo-gw.readthedocs.io/en/latest/waveform_dataset.html) of the DINGO documentation.

For more details on how to parallelize this step over multiple cores on a `htcondor` cluster, you can find more information in one of the [production tutorials](https://dingo-gw.readthedocs.io/en/latest/example_npe_model.html).

In [ ]:
waveform_dataset_settings = """
domain:
  type: UniformFrequencyDomain
  f_min: 20.0
  f_max: 256.0
  delta_f: 0.5 # Expressions like 1.0/8.0 would require eval and are not supported

waveform_generator:
  approximant: IMRPhenomPv2
  f_ref: 20.0
  spin_conversion_phase: 0.0   # Reference phase when converting from spin angles to Cartesian spins. If None, use phase parameter.
  # f_start: 15.0  # Optional setting useful for EOB waveforms. Overrides f_min when generating waveforms.

# Dataset only samples over intrinsic parameters. Extrinsic parameters are chosen at train time.
intrinsic_prior:
  mass_1: bilby.core.prior.Constraint(minimum=20.0, maximum=40.0, name='mass_1')
  mass_2: bilby.core.prior.Constraint(minimum=20.0, maximum=40.0, name='mass_2')
  chirp_mass: bilby.gw.prior.UniformInComponentsChirpMass(minimum=15.0, maximum=50.0, name='chirp_mass')
  mass_ratio: bilby.gw.prior.UniformInComponentsMassRatio(minimum=0.125, maximum=1.0, name='mass_ratio')
  theta_jn: 2.624497413635254
  tilt_1: 2.0111560821533203
  tilt_2: 1.0743615627288818
  a_1: 0.925635814666748
  a_2: 0.5538952350616455
  phi_jl: 5.561878204345703
  phase: 0.9604566579018894
  # Reference values for fixed (extrinsic) parameters. These are needed to generate a waveform.
  luminosity_distance: 100.0  # Mpc
  geocent_time: 0.0 # s

# Dataset size
num_samples: 10_000

compression: None
"""
waveform_dataset_settings = yaml.safe_load(waveform_dataset_settings)
with open('01_training_data/waveform_dataset/waveform_dataset_settings.yaml', 'w') as outfile:
    yaml.dump(waveform_dataset_settings, outfile, default_flow_style=False)


**Step 2: Generate waveform dataset**

To generate the waveform dataset, execute the following command:

In [ ]:
!dingo_generate_dataset --settings 01_training_data/waveform_dataset/waveform_dataset_settings.yaml --out_file 01_training_data/waveform_dataset/waveform_dataset.hdf5

Load the dataset and visualize an exemplary waveform

In [ ]:
from dingo.gw.dataset.waveform_dataset import WaveformDataset

# Load dataset
waveform_dataset_path = '01_training_data/waveform_dataset/waveform_dataset.hdf5'
wfd = WaveformDataset(file_name=waveform_dataset_path)
print("Each element wfd[i] contains an intrinsic waveform. It is a nested dict with structure")
for k, v in wfd[0].items():
    print("  - " + k)
    if isinstance(v, dict):
        for k1 in v:
            print("    - " + k1)

In [ ]:
# Plot a sample waveform
f_domain = wfd.domain.sample_frequencies
data_sample = wfd[0]['waveform']

plt.plot(f_domain, data_sample['h_cross'].real, c="blue", label=r'$h_{\times}$ real')
plt.plot(f_domain, data_sample['h_cross'].imag, c="cornflowerblue", label=r'$h_{\times}$ imag')
plt.plot(f_domain, data_sample['h_plus'].real, c="tab:red", label=r'$h_+$ real')
plt.plot(f_domain, data_sample['h_plus'].imag, c="coral", label=r'$h_+$ imag')
plt.xlabel(r"Frequency $f$ [Hz]")
plt.ylabel(r"Polarization $h$")
plt.legend();

## 2. Generating a noise dataset
In gravitational wave data analysis, the noise is assumed to be stationary and Gaussian in each frequency bin with the variance given the amplitude spectral density (ASD). Similar to extrinsic parameters, detector noise is repeatedly sampled during training and added to the simulated signal. This augments the training set with new noise realizations for each epoch, reducing overfitting.
To take into account that the ASD can drift during an observing run, we create an ASD dataset based on stretches of measured noise in the LIGO and Virgo detectors.
For a detailed explanation about the detector noise, please see the [DINGO documentation](https://dingo-gw.readthedocs.io/en/latest/noise_dataset.html) and paper [Dax+2021](https://arxiv.org/abs/2106.12594).

**Step 1: Prepare settings file**
To generate the ASD dataset, we need to prepare a settings file which contains the following information within `dataset_settings`:
- **f_min, f_max** (Optional)
- **f_s**: Sampling rate. This should be at least twice the value of `f_max` expected to be used.
- **time_psd**: The entire length of data from which to estimate a power spectral density (PSD, $\mathrm{ASD} = \sqrt{\mathrm{PSD}}$) using Welch's method. Periodigrams are calculated on segments of this, and then averaged using the median method.
- **T**: The length of each segment on which to take the DFT and calculate a periodigram.
- **window**: Parameters of the window function used before taking DFT of data segments.
- **channels** (Optional): Channels where to download the data from. By default will use open data from [GWOSC](https://gwosc.org).
- **time_gap**: This sets the time that is skipped between consecutive PSD estimates. If set < 0., the time segments overlap.
- **num_psds_max**: If this is set to 0, all available PSDs will be downloaded. For values > 0, a subset of all available PSDs is used.
- **detectors**: This list specifies for which detectors we want to download ASDs. Options: H1, L1, V1.
- **observing_run**: This sets the observing run of the ASDs. Options: O1, O2, O3.

In a DINGO training run where the initial layers of the embedding network are seeded with SVD components, we typically use two ASD datasets: One with a single, fiducial ASD that is fixed during the first training stage, and a second one with all ASDs of the selected observing run. In this tutorial, we will train the network with a single ASD.

More information about the noise modeling with DINGO can be found in the section [Detector Noise](https://dingo-gw.readthedocs.io/en/latest/noise_dataset.html) of the documentation. It also includes examples for `htcondor` settings that allow to run this step on the cluster.

In [ ]:
asd_dataset_settings = """
dataset_settings:
#  f_min: 0         # defaults to 0
#  f_max: 2048      # defaults to f_s/2
  f_s: 4096
  time_psd: 1024
  T: 2.0
  window:
    roll_off: 0.4
    type: tukey
  time_gap: 0 # specifies the time skipped between to consecutive PSD estimates. If set < 0, the time segments overlap
  num_psds_max: 1 # if set > 0, only a subset of all available PSDs will be used
  detectors:
    - H1
    - L1
  observing_run: O1
"""
asd_dataset_settings = yaml.safe_load(asd_dataset_settings)
with open('01_training_data/asd_dataset/asd_dataset_settings.yaml', 'w') as outfile:
    yaml.dump(asd_dataset_settings, outfile, default_flow_style=False)

**Step 2: Generate ASD dataset**

The following downloads the first valid ASD time segment of the selected observing run (recall `num_psds_max: 1`). For the first observing run O1, this is 1126073529 to 1126074553. It then calculates the ASD using Welch median and saves it to an HDF5 file.

In [ ]:
!dingo_generate_asd_dataset --settings_file 01_training_data/asd_dataset/asd_dataset_settings.yaml --data_dir 01_training_data/asd_dataset --out_name 01_training_data/asd_dataset/asd_fiducial_O1.hdf5

However, we are interested in analyzing the first GW event called GW150914. Therefore, we specify the start and end time of this event in `time_segment.pkl` and provide it as an additional argument to `dingo_generate_asd_dataset`:

In [ ]:
time_GW150914 = 1126259462.391 - 0.0114 # maxlog geocent time of GW150914
asd_start_time = time_GW150914 - asd_dataset_settings["dataset_settings"]["T"]
asd_end_time = time_GW150914
time_segments = {
    'H1': [[asd_start_time, asd_end_time]],
    'L1': [[asd_start_time, asd_end_time]]
}

with open('01_training_data/asd_dataset/time_segment_GW150914.pkl', 'wb') as f:
    pickle.dump(time_segments, f)

In [ ]:
!dingo_generate_asd_dataset --settings_file 01_training_data/asd_dataset/asd_dataset_settings.yaml --data_dir 01_training_data/asd_dataset --out_name 01_training_data/asd_dataset/asd_GW150914.hdf5 --time_segments_file 01_training_data/asd_dataset/time_segment_GW150914.pkl

Load and visualize exemplary ASD:

In [ ]:
from dingo.gw.noise.asd_dataset import ASDDataset

# Load ASD dataset
asd_dataset_path = '01_training_data/asd_dataset/asd_GW150914.hdf5'
asds = ASDDataset(file_name=asd_dataset_path, domain_update=wfd.domain.domain_dict)
# Get ASD sample
asd_sample = asds.sample_random_asds()
print("One Datapoint contains:", asd_sample.keys())

plt.loglog(f_domain, asd_sample['H1'], label=r'H1')
plt.loglog(f_domain, asd_sample['L1'], label=r'L1')
plt.xlabel(r"Frequency [Hz]")
plt.ylabel(r"Noise ASD $[1/\sqrt{\mathrm{Hz}}]$")
plt.xlim([wfd.domain.f_min, wfd.domain.f_max])
plt.ylim([1.e-24, 1.e-20])
plt.legend();

## 3. Training a simple model
**Step 1: Prepare config file**

The train settings YAML file has several sections:

**(a) data**: Specifies training data.
- **waveform_dataset_path**: Path to waveform dataset containing intrinsic waveforms
- **train_fraction**: Fraction of waveform dataset to be used for training. The remainder are used to compute the test loss.
- **detectors**: List of detectors that should be used during training. Options: H1, L1, V1. Note: models with the standard (ResNet) embedding network are trained for a *fixed set of detectors*, so separate models must be trained for, e.g., HL and HLV configurations. Transformer models (DINGO-T1, Section 6) can be trained with detector masking, so that a single model analyzes any subset of its detectors.
- **extrinsic_prior**: Specify the extrinsic prior. Default options are available.
- **ref_time**: Reference time for the interferometer locations and orientations. See the important note [here](https://dingo-gw.readthedocs.io/en/latest/training_transforms.html#ref-ref-time).
- **inference_parameters**: Parameters to infer with the model. Must be a subset of `sample["parameters"]`. By specifying a strict subset, this can be used to marginalize over parameters. The default setting points to `dingo.gw.prior.default_inference_parameters`.

In [ ]:
from dingo.gw.prior import default_inference_parameters
default_inference_parameters


**(b) model**: Specifies neural model.
- **posterior_model_type**. Usually `normalizing_flow` (alternatives `flow_matching` and `score_matching`).
- **posterior_kwargs**: Specifies the density estimator. For `normalizing_flow` this includes the number of flow steps and kwargs of flow transform `base_transform_kwargs`.
- **embedding_type**: `resnet` (residual network whose first layer is initialized with an SVD of the waveforms) or `transformer` (the DINGO-T1 architecture).
- **embedding_kwargs**: Specifies embedding network. This is used to compress the data to a latent representation, which is a passed as context to the posterior model. Arguments include, e.g., dimension of output vector `output_dim`, hidden dimensions `hidden_dims`, the normalization layer `norm` (`BatchNorm` or `LayerNorm`), and how to obtain the SVD components for the initialization of the first layer `svd`.

**(c) training**: Specifies training procedure, optionally divided into stages. Each stage contains:
- **epochs**: Number of epochs
- **asd_dataset_path**: Path to the ASD dataset. If the DINGO model is trained in two stages, this should just contain a single fiducial ASD per detector for the first stage and a full ASD dataset per detector for the second stage.
- **freeze_rb_layer**: Whether to freeze the reduced basis layer in the embedding network in this stage.
- **optimizer**: Optimizer setings.
- **scheduler**: Scheduler settings.
- **batch_size**: Number of data points to use per batch.


**(d) local**: Technical settings (no influence on final model).
- **device**: Which device to train on. Options: `cpu`, `cuda` for training on GPU.
- **num_workers**: Number of workers that are used to preprocess the data before training. (`num_workers >0` does not work on Mac, see [post](https://stackoverflow.com/questions/64772335/pytorch-w-parallelnative-cpp206))
- **runtime_limits**: Specifies the maximum time per run and the maximum number of epochs per run.
- **checkpoint_epochs**: Sets after how many epochs a checkpoint is saved.
- **num_gpus** (optional): Set to >1 for multi-GPU (DDP) training; see [multi-GPU training](https://dingo-gw.readthedocs.io/en/latest/training_multi_gpu.html).

For more information on typical values to be used for production, see the tutorials for [training an NPE model](https://dingo-gw.readthedocs.io/en/latest/example_npe_model.html), [training a GNPE model](https://dingo-gw.readthedocs.io/en/latest/example_gnpe_model.html). The documentation contains more details about the [network architecture](https://dingo-gw.readthedocs.io/en/latest/network_architecture.html) as well as the [training procedure](https://dingo-gw.readthedocs.io/en/latest/training.html).

Since we want to infer the posterior masses of GW150914, we fix the extrinsic prior values to the maximum likelihood values of the infered parameters.

In [ ]:
train_settings = """
data:
  waveform_dataset_path: 01_training_data/waveform_dataset/waveform_dataset.hdf5  # Contains intrinsic waveforms
  train_fraction: 0.95
  detectors:
    - H1
    - L1
  extrinsic_prior:  # Sampled at train time
    dec: bilby.core.prior.analytical.DeltaFunction(-1.2616009712219238)
    ra: bilby.core.prior.analytical.DeltaFunction(1.4557750225067139)
    geocent_time: bilby.core.prior.analytical.DeltaFunction(0.011423417367041111)
    psi: bilby.core.prior.analytical.DeltaFunction(1.2124483585357666)
    luminosity_distance: bilby.core.prior.analytical.DeltaFunction(488.2327880859375)
  ref_time: 1126259462.391
  inference_parameters:
  - chirp_mass
  - mass_ratio

# Model architecture
model:
  posterior_model_type: normalizing_flow
  # kwargs for neural spline flow
  posterior_kwargs:
    num_flow_steps: 5 # 30
    base_transform_kwargs:
      hidden_dim: 64 # 1024
      num_transform_blocks: 5
      activation: elu
      dropout_probability: 0.0
      norm: BatchNorm
      num_bins: 8
      base_transform_type: rq-coupling
  # embedding network
  embedding_type: resnet
  embedding_kwargs:
    output_dim: 64 # 128
    hidden_dims: [1024, 512, 256, 64]
    activation: elu
    dropout: 0.0
    norm: BatchNorm
    svd:
      num_training_samples: 1000
      num_validation_samples: 100
      size: 50

# The first stage (and only) stage of training.
training:
  stage_0:
    epochs: 15
    asd_dataset_path: 01_training_data/asd_dataset/asd_GW150914.hdf5
    freeze_rb_layer: True
    optimizer:
      type: adam
      lr: 0.0001
    scheduler:
      type: cosine
      T_max: 15
    batch_size: 512
  # stage_1:
  #   epochs: 5
  #   asd_dataset_path: 01_training_data/asd_dataset/asd_GW150914.hdf5
  #   freeze_rb_layer: False
  #   optimizer:
  #     type: adam
  #     lr: 1.e-5
  #   scheduler:
  #     type: cosine
  #     T_max: 5
  #   batch_size: 64

# Local settings for training that have no impact on the final trained network.
local:
  device: cpu  # [cpu, cuda] Set this to 'cuda' for training on a GPU.
  num_workers: 0  # set to e.g. 4 on Linux/Colab; num_workers >0 does not work on Mac, see https://stackoverflow.com/questions/64772335/pytorch-w-parallelnative-cpp206
  runtime_limits:
    max_time_per_run: 36000
    max_epochs_per_run: 30
  checkpoint_epochs: 15
"""
train_settings = yaml.safe_load(train_settings)
with open('02_training/train_settings.yaml', 'w') as outfile:
    yaml.dump(train_settings, outfile, default_flow_style=False)

**Step 2: Start training run.**

The following command initiates training. Various pieces of information are output during training, including  SVD performance (e.g., the truncation of the SVD basis and the resulting mismatch), how the new model is initialized, the number of fixed and learnable parameters of the model, the epoch number with training and validation loss, etc.

Training for 15 epochs takes about a minute on a laptop CPU.

In [ ]:
!dingo_train --settings_file 02_training/train_settings.yaml --train_dir 02_training

The loss values are stored in `history.txt`, which contains as columns: epoch number, training loss, validation loss, and learning rate.

One can load this file and plot the training and validation loss:

In [ ]:
# Load history.txt
filename = '02_training/history.txt'
data = np.loadtxt(filename, delimiter="\t")

# Plot loss values
plt.plot(data[:,0], data[:,1], label=f"training loss")
plt.plot(data[:,0], data[:,2], label=f"validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend();

## 4. Injections

Here, we use the **Sampler API** to [prepare and run an injection](https://dingo-gw.readthedocs.io/en/latest/example_injection.html).
1. Load the trained model with `dingo.core.posterior_models.build_model_from_kwargs`.
2. Instantiate the `dingo.gw.injection.Injection` class based on the settings the model was trained on (accessible via `model.metadata`).
3. Specify an ASD dataset, e.g. the fiducial ASD dataset the network was trained on.
4. Sample from the prior and generate an injection.
5. Build a `dingo.gw.inference.sampler.GWComposedSampler` from the model and the injected data. The sampler represents the posterior as a [chain of sampling steps](https://dingo-gw.readthedocs.io/en/latest/sampling_chains.html).
6. Generate samples and convert the sampler instance into a `dingo.gw.result.Result`.
7. Run importance sampling on the result object.
8. Visualize the corner plot and save the result.

This approach is designed to produce injections entirely consistent with training data.

In [ ]:
from dingo.core.posterior_models import build_model_from_kwargs
from dingo.gw.inference.sampler import GWComposedSampler
from dingo.gw.injection import Injection

In [ ]:
model_path = "02_training/model_latest.pt"
asd_path = "01_training_data/asd_dataset/asd_GW150914.hdf5"

# Load the trained network
model = build_model_from_kwargs(filename=model_path, device="cpu", load_training_info=False)

# Generate an injection consistent with the data the model was trained on.
injection = Injection.from_posterior_model_metadata(model.metadata)
injection.asd = ASDDataset(asd_path, ifos=["H1", "L1"])
theta = injection.prior.sample()  # Random injection from prior.
strain_data = injection.injection(theta)

# Generate 10,000 samples from the DINGO model based on the generated injection data.
sampler = GWComposedSampler.from_model(model, strain_data)
sampler.run_sampler(num_samples=10_000, batch_size=5_000)
result = sampler.to_result()

# Importance sample the result.
result.importance_sample(num_processes=8)

# Make a corner plot and save the result. The injected parameters are stored with
# the result and are marked in the corner plot.
result.print_summary()
kwargs = {"legend_font_size": 15, "truth_color": "black"}
parameters = ['chirp_mass', 'mass_ratio']
result.plot_corner(parameters=parameters,
                   filename="03_inference/injection/corner.pdf",
                   **kwargs)
result.to_file("03_inference/injection/result.hdf5")

Since we trained the model in a simplified setting (small waveform dataset, small model, short training), we don't expect high sample efficiencies (> 1%).

## 5. Real events with `dingo_pipe`

Dingo includes a command-line tool [`dingo_pipe`](https://dingo-gw.readthedocs.io/en/latest/dingo_pipe.html) for automating inference tasks. This is based very closely on the [`bilby_pipe`](https://lscsoft.docs.ligo.org/bilby_pipe/master/user-interface.html) package, with suitable modifications.

**Step 1: Prepare `.ini` file for `dingo_pipe`.**
Similar to `bilby_pipe`, `dingo_pipe` requires an `.ini` file as an input that specifies the arguments of the subsequently performed steps.
1. **Job submission arguments**: Specify cluster and system specific details here.
2. **Sampler arguments**: Determine which model should be used to generate the samples.
3. **Data generation arguments**: Define which event data should be downloaded.
4. **Plotting arguments**: Specify which plots should be generated automatically.

In [ ]:
dingo_pipe_GW150914 = """
################################################################################
##  Job submission arguments
################################################################################

local = True
submit = False
request-cpus-importance-sampling = 2
simple-submission = False

################################################################################
##  Sampler arguments
################################################################################

model = 02_training/model_latest.pt
device = cpu
num-samples = 5000
batch-size = 5000
importance-sampling-settings = {}

################################################################################
## Data generation arguments
################################################################################

trigger-time = 1126259462.3885767 # GW150914 # condition network on maxlog geocent time 1126259462.4 - 0.011423417367041111
label = GW150914
outdir = 03_inference/outdir_GW150914
channel-dict = {H1:GWOSC, L1:GWOSC}
psd-length = 128
# sampling-frequency = 2048.0
# importance-sampling-updates = {'duration': 4.0}

################################################################################
## Plotting arguments
################################################################################

plot-corner = true
plot-weights = true
plot-log-probs = true
"""
with open('03_inference/GW150914.ini', 'w') as outfile:
    outfile.write(dingo_pipe_GW150914)

**Step 2: Run `dingo_pipe` for GW150914**

If `local=True`, you can run all steps of `dingo_pipe` automatically by executing the following command:

In [ ]:
!dingo_pipe 03_inference/GW150914.ini

This script produces several files in the folder `03_inference/outdir_GW150914`, where `results` contains the generated (importance) samples, and different plots.

The officially reported values are: chirp mass $\mathcal{M}_c = 27.6^{+2.0}_{-2.0} M_{⊙}$, and mass ratio $q = 0.82^{+0.17}_{-0.20}$ (from [GWOSC](https://gwosc.org/events/GW150914/#P150914), [arXiV:1602.03837](https://arxiv.org/abs/1602.03837)). Similar to the injections, we don't expect good results from this simplified toy setting.

In [ ]:
from dingo.gw.result import Result

# Adjust path as needed
result = Result(file_name="03_inference/outdir_GW150914/result/GW150914_data0_1126259462-3885767_importance_sampling.hdf5")
result.samples.describe()

In [ ]:
result.print_summary()

**Exercises**:
1. Replace the delta function over the luminosity distance in the prior of the DINGO model by a Uniform distribution over $d_L \in [100, 1000]$ Mpc.
2. Train a marginalized model & perform inference.
3. Train a model that infers the luminosity distance as well.

You can use the folder `04_exercise` and its subfolders to save the settings files and models.

# 6. Inference with the pre-trained DINGO-T1 model (from Zenodo)

Larger networks are more expensive to train, often ~ week on an A100. We therefore supply pre-trained networks on Zenodo.

[DINGO-T1](https://dingo-gw.readthedocs.io/en/latest/example_dingo_t1.html) is a transformer-based DINGO model. Instead of compressing the strain with an SVD projection and a residual network, it splits the data in each detector into *tokens* (chunks of frequency bins) and processes them with a transformer encoder. During training, random detectors, frequency ranges, and narrow frequency bands are masked out. As a result, a single model can analyze an event
- with any subset of the detectors H1, L1, V1,
- with a per-detector frequency range (`minimum-frequency`, `maximum-frequency`),
- with PSD notches (`psd-notch-dict`),

without retraining. DINGO-T1 was trained for the third observing run (O3) using the `IMRPhenomXPHM` waveform model. See [Kofler et al., arXiv:2512.02968](https://arxiv.org/abs/2512.02968) and the [DINGO-T1 repository](https://github.com/dingo-gw/dingo-T1) for details.

**Step 1: Download DINGO-T1 from Zenodo**

The model is available on [Zenodo](https://zenodo.org/records/17726076) (641 MB). The download might take some time depending on the internet connection.

In [ ]:
import urllib.request

model_t1 = "05_pretrained_model/dingo_t1.pt"
if not os.path.exists(model_t1):
    urllib.request.urlretrieve("https://zenodo.org/records/17726076/files/dingo_t1.pt?download=1", model_t1)
print(f"{os.path.getsize(model_t1) / 1e6:.0f} MB")

**Step 2: Prepare INI files**

We analyze GW190701_203306, an O3 event observed by all three detectors, and replicate an example from the DINGO-T1 paper: the *same* network analyzes the event once with only LIGO Hanford (H1) and once with both LIGO detectors (H1, L1). The detectors are selected through `channel-dict`; the frequency range through `minimum-frequency` and `maximum-frequency`.

To keep the run time short, we draw 10,000 samples. After sampling, `dingo_pipe` importance samples the result, which includes generating the phase synthetically (this is the most expensive step on a CPU).

In [ ]:
def t1_ini(detectors):
    channels = ", ".join(f"{d}: GWOSC" for d in detectors)
    label = "_".join(detectors)
    return f"""
################################################################################
##  Job submission arguments
################################################################################

local = True
submit = False
request-cpus-importance-sampling = 8
simple-submission = False

################################################################################
##  Sampler arguments
################################################################################

model = 05_pretrained_model/dingo_t1.pt
device = cpu
num-samples = 10_000
batch-size = 5_000

################################################################################
## Data generation arguments
################################################################################

trigger-time = 1246048404.56
label = GW190701_203306
outdir = 05_pretrained_model/GW190701_203306/{label}
channel-dict = {{{channels}}}
psd-length = 64  # in units of the 8 s segment duration
psd-start-time = 8  # relative to the segment start; L1 data begin only 169 s before the event, so use data after it
minimum-frequency = 20.0
maximum-frequency = 448.0

################################################################################
## Plotting arguments
################################################################################

plot-corner = true
plot-weights = true
plot-log-probs = true
"""

for detectors in [["H1"], ["H1", "L1"]]:
    with open(f"05_pretrained_model/GW190701_203306_{'_'.join(detectors)}.ini", "w") as f:
        f.write(t1_ini(detectors))

**Step 3: Run `dingo_pipe` with DINGO-T1**

We run `dingo_pipe` twice, with the same network, once for each detector configuration.

In [ ]:
!dingo_pipe 05_pretrained_model/GW190701_203306_H1.ini

In [ ]:
!dingo_pipe 05_pretrained_model/GW190701_203306_H1_L1.ini

**Step 4: Compare the posteriors**

Adding L1 tightens the posterior: in our run, the 90% intervals of the chirp mass and the luminosity distance shrink by about 40%, and that of the declination by half. The plot shows the importance-weighted samples.

In [ ]:
import glob
import corner
from dingo.gw.result import Result

results_t1 = {}
for label in ["H1", "H1_L1"]:
    fname = glob.glob(f"05_pretrained_model/GW190701_203306/{label}/result/*importance_sampling.hdf5")[0]
    results_t1[label] = Result(file_name=fname)
    results_t1[label].print_summary()

params = ["chirp_mass", "mass_ratio", "luminosity_distance", "ra", "dec"]
# Common axis ranges, so that both posteriors are fully shown
ranges = [(min(r.samples[p].min() for r in results_t1.values()),
           max(r.samples[p].max() for r in results_t1.values())) for p in params]
fig = None
for (label, res), color in zip(results_t1.items(), ["tab:orange", "tab:blue"]):
    samples = res.samples
    fig = corner.corner(samples[params].to_numpy(), weights=samples["weights"].to_numpy(),
                        labels=params, range=ranges, color=color, fig=fig, plot_datapoints=False,
                        plot_density=False, levels=(0.5, 0.9), smooth=1.0)
for label, color in zip(results_t1, ["tab:orange", "tab:blue"]):
    fig.axes[0].plot([], [], color=color, label=label.replace("_", ""))
fig.axes[0].legend(loc="upper left", bbox_to_anchor=(2.5, 1.0), frameon=False);

**Exercises:**
1. Add Virgo (`V1`) to `channel-dict` and compare the three posteriors.
2. Raise `minimum-frequency` for one detector, e.g. `minimum-frequency = {H1: 20, L1: 40}`.
3. Notch a narrow band with `psd-notch-dict = {H1: [59.0, 61.0]}` (see the [DINGO-T1 example](https://dingo-gw.readthedocs.io/en/latest/example_dingo_t1.html)).
4. Run an injection with DINGO-T1 using the Sampler API from Section 4.